# 01 · Data pull

Everything external comes in here and gets cached to `data/`. Later notebooks read the cache, not the internet.

Three inputs:
1. **The Treasury curve.** What the US government pays to borrow at each maturity. It's the "risk-free" baseline: any yield above it is the market charging for credit risk (plus some liquidity).
2. **ICE BofA option-adjusted spreads (OAS) by rating.** The average extra yield over Treasuries that bonds of each rating bucket pay. *Option-adjusted* means callable bonds get corrected for the issuer's right to repay early, so the number is close to pure credit compensation. This is the yardstick: if a BBB bond pays far more than the BBB average, the market thinks it's riskier than its rating says.
3. **`bonds.csv` and `quotes.csv`.** Terms (coupon, maturity, collateral, guarantees) and dated, hand-entered market quotes. Every quote carries an as-of date and a source URL.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Load FRED_API_KEY from .env (python-dotenv isn't installed; this is all it would do)
if Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

DATA = Path('data/fred'); DATA.mkdir(parents=True, exist_ok=True)

TREASURY = ['DGS1', 'DGS2', 'DGS3', 'DGS5', 'DGS7', 'DGS10', 'DGS20', 'DGS30']
OAS = {
    'BAMLC0A0CM': 'IG master', 'BAMLC0A1CAAA': 'AAA', 'BAMLC0A2CAA': 'AA', 'BAMLC0A3CA': 'A', 'BAMLC0A4CBBB': 'BBB',
    'BAMLH0A0HYM2': 'HY master', 'BAMLH0A1HYBB': 'BB', 'BAMLH0A2HYB': 'B', 'BAMLH0A3HYC': 'CCC & lower',
    # IG spreads by maturity (all ratings), for the maturity adjustment in 02_spreads
    'BAMLC1A0C13Y': 'IG 1-3y', 'BAMLC2A0C35Y': 'IG 3-5y', 'BAMLC3A0C57Y': 'IG 5-7y',
    'BAMLC4A0C710Y': 'IG 7-10y', 'BAMLC7A0C1015Y': 'IG 10-15y', 'BAMLC8A0C15PY': 'IG 15y+',
}

### Fetch with a cache that only grows

FRED serves ICE spread history for only the last ~3 years, and that window keeps sliding. So each pull is **merged into** the existing cache file rather than overwriting it. Old rows we already have are never dropped.

If `FRED_API_KEY` is set (in the environment or `.env`) we use `fredapi`; otherwise FRED's public CSV download (no key needed).

In [2]:
def _download(sid):
    key = os.environ.get('FRED_API_KEY')
    if key:
        from fredapi import Fred
        return Fred(api_key=key).get_series(sid).rename(sid)
    df = pd.read_csv(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}', index_col=0, parse_dates=True, na_values='.')
    return df[sid]

def fred(sid, refresh=False):
    path = DATA / f'{sid}.csv'
    old = pd.read_csv(path, index_col=0, parse_dates=True)[sid] if path.exists() else pd.Series(name=sid, dtype=float)
    if path.exists() and not refresh:
        return old
    new = _download(sid).dropna()
    merged = pd.concat([old, new]).groupby(level=0).last().sort_index()  # new values win on overlap
    merged.rename_axis('date').to_csv(path)
    return merged

REFRESH = False  # flip to True to pull fresh data
ust = pd.concat([fred(s, REFRESH) for s in TREASURY], axis=1)
oas = pd.concat([fred(s, REFRESH) for s in OAS], axis=1).rename(columns=OAS)
print(f'Treasury: {ust.index.min():%Y-%m-%d} → {ust.index.max():%Y-%m-%d}')
print(f'OAS:      {oas.index.min():%Y-%m-%d} → {oas.index.max():%Y-%m-%d}')

Treasury: 1962-01-02 → 2026-11-09
OAS:      2023-10-24 → 2026-11-09


/var/folders/wm/gm9z3b_j4cjb_n2bg_3ydpy80000gn/T/ipykernel_88779/3823574888.py:21: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  oas = pd.concat([fred(s, REFRESH) for s in OAS], axis=1).rename(columns=OAS)


### Where things stand today

Yields and spreads are in **percent**. Multiply a spread by 100 for basis points (bp): 0.75% = 75bp.

In [3]:
snap = pd.DataFrame({'Treasury yield (%)': ust.ffill().iloc[-1]})
snap.index = [s.replace('DGS', '') + 'yr' for s in snap.index]
display(snap.T.round(2))
display(oas.ffill().iloc[[-1]].T.rename(columns=lambda d: f'OAS % on {d:%Y-%m-%d}').round(2))

,1yr,2yr,3yr,5yr,7yr,10yr,20yr,30yr
Treasury yield (%),4.38,4.56,4.58,4.67,4.76,4.88,5.29,5.26


date,OAS % on 2026-11-09
IG master,0.91
AAA,0.46
AA,0.67
A,0.75
BBB,1.13
HY master,3.34
BB,2.00
B,3.50
CCC & lower,13.11
IG 1-3y,0.62


## Bond terms and quotes

`bonds.csv` holds terms only: things that don't change after issue. `quotes.csv` holds market observations: price, yield, or spread on a date, with the source. `kind` is `issue` (the term-sheet pricing) or `trade` (secondary market).

Checks below: every quote points to a known bond and has a date and a source, and every bond has a source for its terms.

In [4]:
bonds = pd.read_csv('bonds.csv', parse_dates=['maturity'])
quotes = pd.read_csv('quotes.csv', parse_dates=['as_of'])

assert bonds.bond_id.is_unique
assert bonds.terms_source.notna().all(), 'every bond needs a terms source'
assert quotes.bond_id.isin(bonds.bond_id).all(), set(quotes.bond_id) - set(bonds.bond_id)
assert quotes[['as_of', 'source_url']].notna().all().all(), 'every quote needs a date and a source'

bonds['years_left'] = ((bonds.maturity - pd.Timestamp.today()).dt.days / 365.25).round(1)
display(bonds[['bond_id', 'issuer_entity', 'seniority', 'coupon', 'maturity', 'years_left', 'rating_sp', 'rating_moodys', 'issue_size_usd_bn']])

,bond_id,issuer_entity,seniority,coupon,maturity,years_left,rating_sp,rating_moodys,issue_size_usd_bn
0,CRWV_DDTL4_2032,"CoreWeave Compute Acquisition Co. VIII, LLC",senior secured,5.900,2032-03-31,5.4,NaN,A3,8.500
1,CRWV_9.25_2030,"CoreWeave, Inc.",senior unsecured,9.250,2030-06-01,3.6,B+ (issuer),Ba3 (issuer),2.000
2,CRWV_9.00_2031,"CoreWeave, Inc.",senior unsecured,9.000,2031-02-01,4.2,B+ (issuer),Ba3 (issuer),1.750
3,CRWV_9.75_2031,"CoreWeave, Inc.",senior unsecured,9.750,2031-10-01,4.9,B+ (issuer),Ba3 (issuer),2.750
4,CRWV_9.625_2032,"CoreWeave, Inc.",senior unsecured,9.625,2032-07-15,5.7,B+ (issuer),Ba3 (issuer),1.250
5,META_4.55_2031,"Meta Platforms, Inc.",senior unsecured,4.550,2031-05-15,4.5,AA-,Aa3,3.000
6,META_5.25_2036,"Meta Platforms, Inc.",senior unsecured,5.250,2036-05-15,9.5,AA-,Aa3,6.000
7,META_6.30_2056,"Meta Platforms, Inc.",senior unsecured,6.300,2056-05-15,29.5,AA-,Aa3,6.000
8,META_5.55_2064,"Meta Platforms, Inc.",senior unsecured,5.550,2064-08-15,37.8,AA-,Aa3,2.750
9,BEIGNET_6.581_2049,Beignet Investor LLC,senior secured,6.581,2049-05-30,22.5,A+,NaN,27.294


In [5]:
latest = quotes.sort_values('as_of').groupby('bond_id').tail(1)
stale = latest[latest.as_of < pd.Timestamp.today() - pd.Timedelta(days=14)]
print(f'{len(stale)} of {len(latest)} bonds have no quote in the last 14 days (refresh from TRACE):')
display(stale[['bond_id', 'as_of', 'kind']])
display(bonds.loc[~bonds.bond_id.isin(quotes.bond_id), ['bond_id']].assign(status='no quote at all'))

0 of 21 bonds have no quote in the last 14 days (refresh from TRACE):


,bond_id,as_of,kind


,bond_id,status
0,CRWV_DDTL4_2032,no quote at all


## Stock prices (for 05 · equity vs. bond)

Daily closes from Yahoo Finance's public chart endpoint, cached to `data/equity/{ticker}.csv` with the same only-grows merge as FRED. `close` is the traded price (for market cap). `adjclose` adjusts for splits and dividends (for volatility).

Tickers come from `issuers.csv`. Shares, debt, and cash there are hand-entered from 10-Qs, each with a source URL and as-of date.

In [6]:
import requests
EQ = Path('data/equity'); EQ.mkdir(parents=True, exist_ok=True)

def _yahoo(ticker):
    r = requests.get(f'https://query1.finance.yahoo.com/v8/finance/chart/{ticker}', params={'range': '5y', 'interval': '1d'},
                     headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    res = r.json()['chart']['result'][0]
    idx = pd.to_datetime(res['timestamp'], unit='s').normalize()
    q = res['indicators']
    return pd.DataFrame({'close': q['quote'][0]['close'], 'adjclose': q['adjclose'][0]['adjclose']}, index=idx).dropna()

def equity(ticker, refresh=False):
    path = EQ / f'{ticker}.csv'
    old = pd.read_csv(path, index_col=0, parse_dates=True) if path.exists() else pd.DataFrame(columns=['close', 'adjclose'])
    if path.exists() and not refresh:
        return old
    merged = pd.concat([old, _yahoo(ticker)]).groupby(level=0).last().sort_index()
    merged.rename_axis('date').to_csv(path)
    return merged

issuers = pd.read_csv('issuers.csv')
assert issuers[['source_url', 'balance_date', 'shares_asof']].notna().all().all()
px = {t: equity(t, REFRESH) for t in issuers.ticker}
pd.DataFrame({t: {'from': d.index.min().date(), 'to': d.index.max().date(), 'last close': d.close.iloc[-1]} for t, d in px.items()}).T

,from,to,last close
AAPL,2021-10-25,2026-11-10,348.51001
GOOGL,2021-10-25,2026-11-10,287.380005
META,2021-10-25,2026-11-10,641.390015
SPCX,2026-06-12,2026-11-10,143.089996
CRWV,2025-03-28,2026-11-10,48.73


## Option chains (for 06)

From [Market Data](https://www.marketdata.app/docs/api/options/chain) on the free plan, which is 24 hours delayed: a pull on Monday returns Friday's close. Each contract returned costs one credit, so we pull only two slices per ticker:
1. **Near the money, every expiry** (`strikeLimit=2`, calls and puts). This gives the implied-volatility term structure: how volatile the options market expects the stock to be over 1 month, 1 year, 2 years.
2. **Every put below the current price, at two expiries** (~1 year and the longest listed, ~Jan 2029). This gives the market-implied odds of a big fall by each date.

Each snapshot is saved as `data/options/{ticker}_{date}.csv` and never overwritten. With `REFRESH = False` we load the latest snapshot on disk. The key is read from `MARKETDATA_TOKEN` (`.env`).

In [7]:
OPT = Path('data/options'); OPT.mkdir(parents=True, exist_ok=True)

def _md(ticker, **params):
    r = requests.get(f'https://api.marketdata.app/v1/options/chain/{ticker}/', params=params,
                     headers={'Authorization': f"Bearer {os.environ['MARKETDATA_TOKEN']}"}, timeout=60)
    j = r.json()
    assert j.get('s') == 'ok', (ticker, params, j.get('errmsg'))
    print(f'{ticker} {params}: {len(j["optionSymbol"])} contracts, credits left today: {r.headers.get("X-Api-Ratelimit-Remaining")}')
    return pd.DataFrame({k: v for k, v in j.items() if isinstance(v, list)})

def options(ticker, refresh=False):
    have = sorted(OPT.glob(f'{ticker}_*.csv'))
    if have and not refresh:
        return pd.read_csv(have[-1], parse_dates=['expiration', 'updated'])
    atm = _md(ticker, expiration='all', strikeLimit=2).assign(slice='atm')
    exps = sorted(pd.to_datetime(atm.expiration, unit='s').dt.normalize().unique())
    one_yr = min(exps, key=lambda e: abs((e - exps[0]).days - 365))
    strips = [_md(ticker, expiration=f'{e:%Y-%m-%d}', side='put', range='otm').assign(slice='put strip') for e in (one_yr, exps[-1])]
    d = pd.concat([atm, *strips], ignore_index=True)
    d['expiration'] = pd.to_datetime(d.expiration, unit='s').dt.normalize()
    d['updated'] = pd.to_datetime(d.updated, unit='s')
    d.to_csv(OPT / f'{ticker}_{d.updated.max():%Y-%m-%d}.csv', index=False)
    return d

chains = {t: options(t, REFRESH) for t in issuers.ticker}
pd.DataFrame({t: {'as of': d.updated.max().date(), 'contracts': len(d), 'expiries': d.expiration.nunique(),
                  'longest': d.expiration.max().date()} for t, d in chains.items()}).T

,as of,contracts,expiries,longest
AAPL,2026-11-09,215,24,2029-01-19
GOOGL,2026-11-09,203,24,2029-01-19
META,2026-11-09,210,24,2029-01-19
SPCX,2026-11-09,160,26,2029-01-19
CRWV,2026-11-09,96,19,2029-01-19


## Company financials from SEC XBRL (for 06)

The SEC's free [XBRL API](https://www.sec.gov/page/sec-api-documentation) returns every number a company has tagged in its 10-K/10-Q filings. That's primary-source data, already in machine-readable form. One JSON per company, cached to `data/edgar/companyfacts_{ticker}.json`.

In [8]:
import json
CIK = {'AAPL': 320193, 'GOOGL': 1652044, 'META': 1326801, 'SPCX': 1181412, 'CRWV': 1769628}
SEC_UA = {'User-Agent': os.environ.get('SEC_USER_AGENT', 'set SEC_USER_AGENT in .env')}

def companyfacts(ticker, refresh=False):
    path = Path(f'data/edgar/companyfacts_{ticker}.json')
    if not path.exists() or refresh:
        r = requests.get(f'https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK[ticker]:010d}.json', headers=SEC_UA, timeout=60)
        r.raise_for_status()
        path.write_text(r.text)
    return json.loads(path.read_text())

facts = {t: companyfacts(t, REFRESH) for t in CIK}
pd.DataFrame({t: {'entity': f['entityName'], 'us-gaap tags': len(f['facts'].get('us-gaap', {}))} for t, f in facts.items()}).T

,entity,us-gaap tags
AAPL,Apple Inc.,503
GOOGL,Alphabet Inc.,543
META,"Meta Platforms, Inc.",462
SPCX,Space Exploration Technologies Corp.,182
CRWV,"CoreWeave, Inc.",305
